# Imports

In [ ]:
import glob

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.model_selection import GroupKFold

from uavloc.matching.benchmark import HARM_MARGIN_M
from uavloc.track import REPO_ROOT

# Params

In [ ]:
BUILD = "corrector-identity-bench"
RECORDS_GLOB = "experiments/021-building-loss-from-scratch/artifacts/correction-records-*.csv"
RULE = "gated"               # s1/s2 depend on `temperature`; only the gated rows use the gate's 0.25
GATE_S_MAX_M = 3.0           # the 007.8 operating point under test
N_FOLDS = 3
SEED = 42

# Data

In [ ]:
OUT = REPO_ROOT / "builds" / BUILD


def load_records(pattern: str, rule: str) -> pd.DataFrame:
    """Every 021 correction draw that the gate actually had to decide on, labelled by whether the argmax harms."""
    paths = sorted(glob.glob(str(REPO_ROOT / pattern)))
    if not paths:
        raise FileNotFoundError(f"no records matched {pattern}")
    frames = [pd.read_csv(p).assign(model=p.split("records-")[1][: -len(".csv")]) for p in paths]
    records = pd.concat(frames, ignore_index=True)
    decidable = records[(records["rule"] == rule) & (records["status"] == "ok") & (records["peak_z"] > 0)].copy()
    decidable["bad"] = decidable["map_error_m"] > decidable["prior_error_m"] + HARM_MARGIN_M
    return decidable

In [ ]:
records = load_records(RECORDS_GLOB, RULE)
{"models": records["model"].nunique(), "draws": len(records),
 "bad_share": round(records["bad"].mean(), 3), "reachable_share": round(records["reachable"].mean(), 3)}

# Model

In [ ]:
def candidates(records: pd.DataFrame) -> dict[str, pd.Series]:
    """Each statistic oriented so that a HIGHER value should mean a more likely harmful correction."""
    return {"s1_m": records["s1_m"], "s2_m": records["s2_m"],
            "second_peak_ratio": records["second_peak_ratio"].fillna(0.0),
            "-peak_z": -records["peak_z"], "-coverage": -records["coverage"]}

In [ ]:
def feature_matrix(records: pd.DataFrame) -> np.ndarray:
    """The five statistics as model inputs; the spread axes are log-compressed, having long right tails."""
    return np.c_[np.log1p(records["s1_m"]), np.log1p(records["s2_m"]),
                 records["second_peak_ratio"].fillna(0.0), records["peak_z"], records["coverage"]]

In [ ]:
def out_of_fold_scores(records: pd.DataFrame, n_folds: int) -> np.ndarray:
    """Combined score, grouped by frame so no frame is in both train and test."""
    features, target = feature_matrix(records), records["bad"].to_numpy()
    scores = np.zeros(len(records))
    for train, test in GroupKFold(n_folds).split(features, target, groups=records["frame"]):
        model = LogisticRegression(max_iter=2000, random_state=SEED).fit(features[train], target[train])
        scores[test] = model.predict_proba(features[test])[:, 1]
    return scores

In [ ]:
def false_alarm_at(score: pd.Series, bad: np.ndarray, detection: float) -> float:
    """Share of GOOD draws this statistic would also block, at the threshold that catches `detection` of the bad."""
    threshold = np.quantile(score[bad], 1 - detection)
    return float((score[~bad] >= threshold).mean())

# Run

In [ ]:
bad = records["bad"].to_numpy()
scores = {**candidates(records), "combined": pd.Series(out_of_fold_scores(records, N_FOLDS), index=records.index)}

In [ ]:
accepted = records["s1_m"] <= GATE_S_MAX_M
operating = {"accepts": accepted.mean(), "detection": 1 - accepted[bad].mean(),
             "false_alarm": 1 - accepted[~bad].mean(),
             "blocked_that_were_good": (~bad[~accepted]).mean(), "accepted_that_were_bad": bad[accepted].mean()}
{k: round(v, 3) for k, v in operating.items()}

In [ ]:
comparison = pd.DataFrame({
    name: {"auc": roc_auc_score(bad, score),
           "false_alarm_at_007.8_detection": false_alarm_at(score, bad, operating["detection"])}
    for name, score in scores.items()}).T.round(3).sort_values("auc", ascending=False)
comparison.to_csv(OUT / "runs" / "gate-discriminators.csv")
comparison

In [ ]:
fpr, tpr, thresholds = roc_curve(bad, records["s1_m"])
best = int(np.argmax(tpr - fpr))
{"youden_threshold_m": round(float(thresholds[best]), 2), "detection": round(float(tpr[best]), 3),
 "false_alarm": round(float(fpr[best]), 3), "tuned_threshold_m": GATE_S_MAX_M}

In [ ]:
def policy_stats(records: pd.DataFrame, accept: np.ndarray) -> dict[str, float]:
    """What a policy actually costs: accepted draws take the argmax, blocked draws keep the prior.

    Scored over EVERY draw. A mean over the accepted subset alone flatters any gate, because it drops
    the hard draws the gate refused instead of charging it for keeping their prior.
    """
    prior = records["prior_error_m"].to_numpy()
    error = np.where(accept, records["map_error_m"].to_numpy(), prior)
    return {"mean_m": error.mean(), "median_m": np.median(error), "p90_m": np.percentile(error, 90),
            "max_m": error.max(), "harm_share": (error > prior + HARM_MARGIN_M).mean(),
            "applied_share": accept.mean()}

In [ ]:
policies = {"accept all (no gate)": np.ones(len(records), bool),
            "s1 gate (007.8)": accepted.to_numpy(),
            "keep prior always": np.zeros(len(records), bool)}
worth = pd.DataFrame({name: policy_stats(records, accept) for name, accept in policies.items()}).T.round(3)
worth.to_csv(OUT / "runs" / "gate-worth.csv")
worth

# Plots

In [ ]:
def plot_roc(scores: dict[str, pd.Series], bad: np.ndarray, ax: plt.Axes) -> None:
    for name, score in scores.items():
        x, y, _ = roc_curve(bad, score)
        ax.plot(x, y, label=f"{name} ({roc_auc_score(bad, score):.3f})")
    ax.plot([0, 1], [0, 1], ls=":", c="grey", lw=1)
    ax.set_xlabel("false alarm — good corrections blocked")
    ax.set_ylabel("detection — harmful corrections caught")
    ax.legend(fontsize=8); ax.grid(alpha=.3)

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 5))
plot_roc(scores, bad, ax)
ax.scatter([operating["false_alarm"]], [operating["detection"]], c="red", zorder=5, s=60,
           label=f"007.8 s1 <= {GATE_S_MAX_M:g} m")
ax.legend(fontsize=8)
ax.set_title(f"catching a harmful correction ({len(records)} draws, {RULE})")
fig.tight_layout()
fig.savefig(OUT / "figures" / "gate-roc.png", dpi=150)